# 05 — WildfireQCircuit: Domain-Informed Quantum Model

**Key Innovation:** Instead of using Qiskit's generic `ZZFeatureMap + RealAmplitudes`,
we build a **custom quantum circuit** whose entanglement topology is derived from
wildfire science literature. Each CZ gate corresponds to a known physical or
socioeconomic interaction between features.

## Qubit Layout (Domain Clusters)

| Qubit | Cluster | Feature | Rationale |
|---|---|---|---|
| q0 | Fire History | prior_total_acres | Cumulative burn history |
| q1 | Fire History | prior_max_acres | Peak fire severity |
| q2 | Fire History | Num High Fire Risk Exposure | Current hazard potential |
| q3 | Weather | min_precip | Moisture/drought proxy |
| q4 | Insurance | Earned Premium | Financial risk signal |
| q5 | Insurance | CAT CovA Fire Claims | Catastrophe loss history |
| q6 | Insurance | Avg PPC | Protection classification |
| q7 | Socioeconomic | renter_occupied_units | Housing vulnerability |
| q8 | Socioeconomic | education_bachelor | Income/wealth proxy |
| q9 | Socioeconomic | housing_vacancy | Community stability |

## Entanglement Justification

- **Fire History triangle (q0↔q1↔q2):** Cumulative burn area, peak severity, and current risk classification are tightly coupled (Oliveira et al. 2021, Mhawej et al. 2015)
- **Insurance triangle (q4↔q5↔q6):** Premium pricing depends on claims history and protection class (Auer & Hexamer 2022)
- **Socioeconomic triangle (q7↔q8↔q9):** Renter density, education, and vacancy cluster geographically (Auer & Hexamer 2022)
- **Moisture → Fire History (q3↔q0, q3↔q2):** Precipitation modulates fuel dryness and fire spread (Varga et al. 2022, Mhawej et al. 2015)
- **Premium → Exposure (q4↔q2):** Insurance pricing directly driven by fire exposure counts
- **Demographics → Exposure (q7↔q2, q8↔q2):** Socioeconomic vulnerability amplifies risk exposure (Auer & Hexamer 2022)

In [ ]:
# === Cell 1: Imports and Data Loading ===
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
from imblearn.under_sampling import RandomUnderSampler

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA

# Load data
X_train_top10 = np.load('../data/X_train_top10.npy')
X_test_top10 = np.load('../data/X_test_top10.npy')
y_train = np.load('../data/y_train.npy')
y_test = np.load('../data/y_test.npy')

print(f"Loaded: X_train {X_train_top10.shape}, X_test {X_test_top10.shape}")
print(f"Fire rate — train: {y_train.mean():.2%}, test: {y_test.mean():.2%}")

In [ ]:
# === Cell 2: Reorder Features into Domain Clusters ===
#
# Original column order in X_train_top10:
#   0: Number of High Fire Risk Exposure
#   1: prior_max_acres
#   2: Avg PPC
#   3: Earned Premium
#   4: min_precip
#   5: CAT Cov A Fire - Number of Claims
#   6: prior_total_acres
#   7: renter_occupied_housing_units
#   8: educational_attainment_bachelor_or_higher
#   9: housing_vacancy_number
#
# New order groups features by domain cluster:
#   Fire History (q0-q2) → Insurance (q4-q6) → Socioeconomic (q7-q9)
#   with Weather (q3) bridging Fire History and Insurance

REORDER = [6, 1, 0, 4, 3, 5, 2, 7, 8, 9]

FEATURE_NAMES = [
    'prior_total_acres',                          # q0 - Fire History
    'prior_max_acres',                            # q1 - Fire History
    'Number of High Fire Risk Exposure',          # q2 - Fire History
    'min_precip',                                 # q3 - Weather
    'Earned Premium',                             # q4 - Insurance
    'CAT Cov A Fire - Number of Claims',          # q5 - Insurance
    'Avg PPC',                                    # q6 - Insurance
    'renter_occupied_housing_units',              # q7 - Socioeconomic
    'educational_attainment_bachelor_or_higher',  # q8 - Socioeconomic
    'housing_vacancy_number',                     # q9 - Socioeconomic
]

CLUSTERS = [
    'Fire History', 'Fire History', 'Fire History',
    'Weather',
    'Insurance', 'Insurance', 'Insurance',
    'Socioeconomic', 'Socioeconomic', 'Socioeconomic'
]

X_train_reord = X_train_top10[:, REORDER]
X_test_reord = X_test_top10[:, REORDER]

# Save for reproducibility
np.save('../data/X_train_wildfire_circuit.npy', X_train_reord)
np.save('../data/X_test_wildfire_circuit.npy', X_test_reord)

print("Reordered feature layout:")
print(f"{'Qubit':<6} {'Cluster':<15} {'Feature'}")
print("-" * 65)
for i, (name, cluster) in enumerate(zip(FEATURE_NAMES, CLUSTERS)):
    print(f"q{i:<5} {cluster:<15} {name}")

print(f"\nTrain shape: {X_train_reord.shape}, Test shape: {X_test_reord.shape}")

In [ ]:
# === Cell 3: Prepare Balanced Training Set ===

rus = RandomUnderSampler(random_state=42)
X_bal, y_bal = rus.fit_resample(X_train_reord, y_train)

print(f"Full balanced set: {X_bal.shape[0]} samples ({y_bal.mean():.0%} fire)")
print(f"  No fire: {(y_bal == 0).sum()}, Fire: {(y_bal == 1).sum()}")

In [ ]:
# === Cell 4: Build the WildfireQCircuit Feature Map ===
#
# This is the core innovation. Every entangling gate is motivated by
# wildfire science literature.
#
# Architecture: 2 layers of (H → RY encoding → domain CZ entanglement)
# The second layer implements "data re-uploading" which increases
# the expressiveness of the feature map (Pérez-Salinas et al., 2020).

N_QUBITS = 10

def build_wildfire_feature_map(n_qubits=10, reps=2):
    """Domain-informed feature map for wildfire risk prediction.
    
    Entanglement topology reflects known interactions:
    - Fire History triangle (q0,q1,q2): burn history correlations
    - Insurance triangle (q4,q5,q6): premium-claims-protection loop
    - Socioeconomic triangle (q7,q8,q9): demographic clustering
    - Moisture→History (q3→q0,q2): fuel-moisture interaction
    - Insurance→Exposure (q4→q2): risk-pricing feedback
    - Demographics→Exposure (q7→q2, q8→q2): vulnerability amplification
    """
    x = ParameterVector('x', n_qubits)
    qc = QuantumCircuit(n_qubits)
    
    for layer in range(reps):
        # --- Encoding: Hadamard + RY rotation per qubit ---
        if layer == 0:
            for i in range(n_qubits):
                qc.h(i)
        for i in range(n_qubits):
            qc.ry(x[i], i)
        
        # --- Within-cluster entanglement (full triangles) ---
        # Fire History: q0↔q1↔q2
        qc.cz(0, 1)  # prior_total ↔ prior_max
        qc.cz(1, 2)  # prior_max ↔ high_risk_exposure
        qc.cz(0, 2)  # prior_total ↔ high_risk_exposure
        
        # Insurance: q4↔q5↔q6
        qc.cz(4, 5)  # premium ↔ claims
        qc.cz(5, 6)  # claims ↔ PPC
        qc.cz(4, 6)  # premium ↔ PPC
        
        # Socioeconomic: q7↔q8↔q9
        qc.cz(7, 8)  # renters ↔ education
        qc.cz(8, 9)  # education ↔ vacancy
        qc.cz(7, 9)  # renters ↔ vacancy
        
        # --- Cross-cluster causal pathways ---
        # Moisture ↔ Fire History (Varga et al. 2022, Mhawej et al. 2015)
        qc.cz(3, 0)  # min_precip ↔ prior_total_acres
        qc.cz(3, 2)  # min_precip ↔ high_risk_exposure
        
        # Insurance ↔ Fire Exposure (Auer & Hexamer 2022)
        qc.cz(4, 2)  # premium ↔ high_risk_exposure
        
        # Socioeconomic ↔ Fire Exposure (Auer & Hexamer 2022)
        qc.cz(8, 2)  # education ↔ high_risk_exposure
        qc.cz(7, 2)  # renters ↔ high_risk_exposure
    
    return qc


def build_wildfire_ansatz(n_qubits=10, n_layers=3):
    """Trainable ansatz with SAME entanglement topology as feature map.
    
    Using the same topology means the trainable parameters learn along
    the same interaction pathways that encode domain knowledge.
    """
    n_params = n_qubits * (n_layers + 1)  # +1 for final rotation layer
    theta = ParameterVector('θ', n_params)
    qc = QuantumCircuit(n_qubits)
    
    for layer in range(n_layers):
        offset = layer * n_qubits
        
        # Trainable RY rotations
        for i in range(n_qubits):
            qc.ry(theta[offset + i], i)
        
        # Same domain-informed entanglement topology
        # Fire History cluster
        qc.cz(0, 1); qc.cz(1, 2); qc.cz(0, 2)
        # Insurance cluster
        qc.cz(4, 5); qc.cz(5, 6); qc.cz(4, 6)
        # Socioeconomic cluster
        qc.cz(7, 8); qc.cz(8, 9); qc.cz(7, 9)
        # Cross-cluster
        qc.cz(3, 0); qc.cz(3, 2)
        qc.cz(4, 2)
        qc.cz(8, 2); qc.cz(7, 2)
    
    # Final rotation layer (no entanglement after)
    offset = n_layers * n_qubits
    for i in range(n_qubits):
        qc.ry(theta[offset + i], i)
    
    return qc


# Build circuits
wildfire_fm = build_wildfire_feature_map(N_QUBITS, reps=2)
wildfire_an = build_wildfire_ansatz(N_QUBITS, n_layers=3)

full_circuit = wildfire_fm.compose(wildfire_an)

print("=== WildfireQCircuit Summary ===")
print(f"Qubits: {N_QUBITS}")
print(f"Feature map parameters: {wildfire_fm.num_parameters} (data features)")
print(f"Ansatz parameters: {wildfire_an.num_parameters} (trainable)")
print(f"Feature map depth: {wildfire_fm.depth()}")
print(f"Ansatz depth: {wildfire_an.depth()}")
print(f"Full circuit depth: {full_circuit.depth()}")
print(f"\nFeature map gates: {dict(wildfire_fm.count_ops())}")
print(f"Ansatz gates: {dict(wildfire_an.count_ops())}")

# Count entangling gates per cluster
print(f"\nEntanglement budget per layer:")
print(f"  Fire History triangle:     3 CZ gates (q0↔q1, q1↔q2, q0↔q2)")
print(f"  Insurance triangle:        3 CZ gates (q4↔q5, q5↔q6, q4↔q6)")
print(f"  Socioeconomic triangle:    3 CZ gates (q7↔q8, q8↔q9, q7↔q9)")
print(f"  Moisture→History:          2 CZ gates (q3↔q0, q3↔q2)")
print(f"  Insurance→Exposure:        1 CZ gate  (q4↔q2)")
print(f"  Demographics→Exposure:     2 CZ gates (q7↔q2, q8↔q2)")
print(f"  Total per layer:          14 CZ gates")
print(f"  vs. ZZFeatureMap linear:   9 CZ gates (only nearest-neighbor)")

In [ ]:
# === Cell 5: Visualize the entanglement topology ===

print("=== Entanglement Topology (Adjacency) ===")
print()
print("  FIRE HISTORY          WEATHER         INSURANCE         SOCIOECONOMIC")
print("  ┌─────────────┐       ┌─────┐        ┌─────────────┐   ┌─────────────┐")
print("  │ q0──q1──q2  │◄─────►│ q3  │        │ q4──q5──q6  │   │ q7──q8──q9  │")
print("  │ └────────┘  │       │     │        │ └────────┘  │   │ └────────┘  │")
print("  │  (triangle) │       └──┬──┘        └──────┬──────┘   └──────┬──────┘")
print("  └──────┬──────┘          │                  │                 │")
print("         │           q3↔q0 (moisture→fuel)    │                 │")
print("         │           q3↔q2 (moisture→risk)    │                 │")
print("         │                                    │                 │")
print("         ├◄──────── q4↔q2 (premium→exposure) ─┘                 │")
print("         │                                                      │")
print("         ├◄──────── q8↔q2 (education→exposure) ────────────────┤")
print("         └◄──────── q7↔q2 (renters→exposure) ─────────────────┘")
print()
print("Key insight: q2 (High Fire Risk Exposure) is the most-connected qubit")
print("because fire exposure is influenced by ALL other domains:")
print("  - Fire history (q0, q1)")
print("  - Moisture/drought (q3)")
print("  - Insurance pricing (q4)")
print("  - Demographics (q7, q8)")
print(f"  → q2 has 6 entangling connections (hub node)")

In [ ]:
# === Cell 6: Evaluation Helper ===

def evaluate_model(name, y_true, y_pred):
    """Print evaluation metrics for a model."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"  TN={cm[0][0]:>4d}  FP={cm[0][1]:>4d}")
    print(f"  FN={cm[1][0]:>4d}  TP={cm[1][1]:>4d}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['No Fire', 'Fire'])}")
    
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

print("Evaluation helper ready.")

---
## Model A: WildfireQCircuit (Domain-Informed)

This is the primary experiment. Custom feature map + custom ansatz,
both using the domain-informed entanglement topology.

**Expected training time: 2-3 hours** (similar to the previous best model
which took 158 min with ZZFeatureMap on 1312 samples × 300 iterations).

In [ ]:
# === Cell 7: MODEL A — WildfireQCircuit (Domain-Informed) ===

print("="*60)
print("  MODEL A: WildfireQCircuit (Domain-Informed Entanglement)")
print("="*60)

# Build circuits
fm_A = build_wildfire_feature_map(N_QUBITS, reps=2)
an_A = build_wildfire_ansatz(N_QUBITS, n_layers=3)

print(f"Feature map: custom WildfireQCircuit (2 encoding layers)")
print(f"Ansatz: custom domain-informed (3 trainable layers + final rotation)")
print(f"Trainable parameters: {an_A.num_parameters}")
print(f"Training data: {X_bal.shape[0]} balanced samples")
print(f"Optimizer: COBYLA, maxiter=300")
print(f"\nStarting training... (this will take 2-3 hours)")
print(f"Start time: {time.strftime('%H:%M:%S')}")

optimizer_A = COBYLA(maxiter=300)

vqc_A = VQC(
    feature_map=fm_A,
    ansatz=an_A,
    optimizer=optimizer_A,
    num_qubits=N_QUBITS,
)

start_A = time.time()
vqc_A.fit(X_bal, y_bal)
time_A = time.time() - start_A

print(f"\nTraining complete in {time_A/60:.1f} minutes")
print(f"End time: {time.strftime('%H:%M:%S')}")

# Predict on full test set
y_pred_A = vqc_A.predict(X_test_reord)
results_A = evaluate_model("Model A: WildfireQCircuit", y_test, y_pred_A)

# Resource requirements
print(f"\n=== Resource Requirements ===")
print(f"Qubits: {N_QUBITS}")
print(f"Feature map depth: {fm_A.depth()}")
print(f"Ansatz depth: {an_A.depth()}")
print(f"Trainable parameters: {an_A.num_parameters}")
print(f"Feature map CZ gates: {dict(fm_A.count_ops()).get('cz', 0)}")
print(f"Ansatz CZ gates: {dict(an_A.count_ops()).get('cz', 0)}")
print(f"Training samples: {X_bal.shape[0]}")
print(f"Optimizer: COBYLA (maxiter=300)")
print(f"Training time: {time_A/60:.1f} minutes")
print(f"Simulator: Qiskit Aer (statevector)")

---
## Model B: Baseline (ZZFeatureMap + RealAmplitudes)

Same features, same reordered data, same training set, same optimizer.
Only difference: uses the **generic** ZZFeatureMap and RealAmplitudes
with linear entanglement (your current best architecture).

This is the **controlled comparison** — if Model A beats Model B,
the improvement is due to the domain-informed topology, not the features.

In [ ]:
# === Cell 8: MODEL B — Baseline (ZZFeatureMap + RealAmplitudes) ===

print("="*60)
print("  MODEL B: Baseline (ZZFeatureMap + RealAmplitudes, linear)")
print("="*60)

fm_B = ZZFeatureMap(feature_dimension=N_QUBITS, reps=2, entanglement='linear')
an_B = RealAmplitudes(num_qubits=N_QUBITS, reps=2, entanglement='linear')

print(f"Feature map: ZZFeatureMap (reps=2, linear entanglement)")
print(f"Ansatz: RealAmplitudes (reps=2, linear entanglement)")
print(f"Trainable parameters: {an_B.num_parameters}")
print(f"Training data: {X_bal.shape[0]} balanced samples")
print(f"Optimizer: COBYLA, maxiter=300")
print(f"\nStarting training...")
print(f"Start time: {time.strftime('%H:%M:%S')}")

optimizer_B = COBYLA(maxiter=300)

vqc_B = VQC(
    feature_map=fm_B,
    ansatz=an_B,
    optimizer=optimizer_B,
    num_qubits=N_QUBITS,
)

start_B = time.time()
vqc_B.fit(X_bal, y_bal)
time_B = time.time() - start_B

print(f"\nTraining complete in {time_B/60:.1f} minutes")

y_pred_B = vqc_B.predict(X_test_reord)
results_B = evaluate_model("Model B: ZZFeatureMap + RealAmplitudes (linear)", y_test, y_pred_B)

print(f"\n=== Resource Requirements ===")
print(f"Trainable parameters: {an_B.num_parameters}")
print(f"Training time: {time_B/60:.1f} minutes")

---
## Model C: Random Entanglement (Ablation Control)

Same number of CZ gates as Model A, but connections are **random**.
Same feature encoding (H + RY), same ansatz structure, same data.

If Model A > Model C, it proves the **specific topology** matters,
not just "having more entanglement than linear."

In [ ]:
# === Cell 9: MODEL C — Random Entanglement (Ablation) ===

def build_random_feature_map(n_qubits=10, n_cz_per_layer=14, reps=2, seed=123):
    """Feature map with SAME structure as WildfireQCircuit but RANDOM CZ pairs.
    Same number of CZ gates per layer (14) to control for entanglement budget."""
    rng = np.random.RandomState(seed)
    x = ParameterVector('x', n_qubits)
    qc = QuantumCircuit(n_qubits)
    
    # Generate random CZ pairs (fixed across layers for fair comparison)
    all_pairs = [(i, j) for i in range(n_qubits) for j in range(i+1, n_qubits)]
    chosen_pairs = [tuple(p) for p in rng.choice(len(all_pairs), n_cz_per_layer, replace=False)]
    random_cz_pairs = [all_pairs[idx] for idx in chosen_pairs]
    
    for layer in range(reps):
        if layer == 0:
            for i in range(n_qubits):
                qc.h(i)
        for i in range(n_qubits):
            qc.ry(x[i], i)
        for (a, b) in random_cz_pairs:
            qc.cz(a, b)
    
    return qc, random_cz_pairs


def build_random_ansatz(n_qubits=10, n_layers=3, cz_pairs=None):
    """Ansatz with same random CZ topology."""
    n_params = n_qubits * (n_layers + 1)
    theta = ParameterVector('θ', n_params)
    qc = QuantumCircuit(n_qubits)
    
    for layer in range(n_layers):
        offset = layer * n_qubits
        for i in range(n_qubits):
            qc.ry(theta[offset + i], i)
        for (a, b) in cz_pairs:
            qc.cz(a, b)
    
    offset = n_layers * n_qubits
    for i in range(n_qubits):
        qc.ry(theta[offset + i], i)
    
    return qc


print("="*60)
print("  MODEL C: Random Entanglement (Ablation Control)")
print("="*60)

fm_C, random_pairs = build_random_feature_map(N_QUBITS, n_cz_per_layer=14, reps=2, seed=123)
an_C = build_random_ansatz(N_QUBITS, n_layers=3, cz_pairs=random_pairs)

print(f"Random CZ pairs (14 per layer): {random_pairs}")
print(f"Trainable parameters: {an_C.num_parameters}")
print(f"CZ gates in feature map: {dict(fm_C.count_ops()).get('cz', 0)}")
print(f"CZ gates in ansatz: {dict(an_C.count_ops()).get('cz', 0)}")
print(f"\nStarting training...")
print(f"Start time: {time.strftime('%H:%M:%S')}")

optimizer_C = COBYLA(maxiter=300)

vqc_C = VQC(
    feature_map=fm_C,
    ansatz=an_C,
    optimizer=optimizer_C,
    num_qubits=N_QUBITS,
)

start_C = time.time()
vqc_C.fit(X_bal, y_bal)
time_C = time.time() - start_C

print(f"\nTraining complete in {time_C/60:.1f} minutes")

y_pred_C = vqc_C.predict(X_test_reord)
results_C = evaluate_model("Model C: Random Entanglement", y_test, y_pred_C)

In [ ]:
# === Cell 10: Ablation Comparison Table ===

print("\n" + "="*80)
print("  ABLATION STUDY: Does Domain-Informed Entanglement Matter?")
print("="*80)
print(f"\n{'Model':<45s} {'Acc':>6s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s} {'Time':>7s}")
print(f"{'-'*80}")

# Previous best (from notebook 04, for reference)
print(f"{'[Ref] VQC Best (ZZ+RA, linear, old order)':<45s} {'0.793':>6s} {'0.203':>6s} {'0.478':>6s} {'0.285':>6s} {'158m':>7s}")

# Current experiments
for name, res, t in [
    ("A: WildfireQCircuit (domain-informed)", results_A, time_A),
    ("B: ZZFeatureMap+RealAmplitudes (linear)", results_B, time_B),
    ("C: Random entanglement (14 CZ/layer)", results_C, time_C),
]:
    print(f"{name:<45s} {res['accuracy']:>6.3f} {res['precision']:>6.3f} {res['recall']:>6.3f} {res['f1']:>6.3f} {t/60:>6.1f}m")

# Classical baselines for context
print(f"{'-'*80}")
print(f"{'[Classical] XGBoost (43 features)':<45s} {'0.889':>6s} {'0.412':>6s} {'0.665':>6s} {'0.509':>6s} {'<1s':>7s}")
print(f"{'[Classical] XGBoost (top-10 features)':<45s} {'0.904':>6s} {'0.372':>6s} {'0.710':>6s} {'0.488':>6s} {'<1s':>7s}")

print(f"\n=== Interpretation ===")
if results_A['f1'] > results_B['f1']:
    improvement = (results_A['f1'] - results_B['f1']) / results_B['f1'] * 100
    print(f"Model A (domain-informed) beats Model B (generic) by {improvement:.1f}% in F1")
else:
    print(f"Model B (generic) performed equal or better than Model A")

if results_A['f1'] > results_C['f1']:
    improvement_c = (results_A['f1'] - results_C['f1']) / results_C['f1'] * 100
    print(f"Model A (domain-informed) beats Model C (random) by {improvement_c:.1f}% in F1")
    print(f"→ The specific entanglement TOPOLOGY matters, not just entanglement density")
else:
    print(f"Model C (random) performed equal or better — topology effect inconclusive")

In [ ]:
# === Cell 11: Full Comparison with All Prior Models ===

print("\n" + "="*85)
print("  COMPLETE MODEL COMPARISON (Task 1A)")
print("="*85)
print(f"\n{'Model':<50s} {'Acc':>6s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s}")
print(f"{'-'*85}")

all_results = [
    ("CLASSICAL BASELINES", None, None, None, None),
    ("  Random Forest (43 features)", 0.860, 0.351, 0.728, 0.473),
    ("  XGBoost (43 features)", 0.889, 0.412, 0.665, 0.509),
    ("  XGBoost (top-10 features)", 0.904, 0.372, 0.710, 0.488),
    ("", None, None, None, None),
    ("QUANTUM — GENERIC CIRCUITS", None, None, None, None),
    ("  VQC 6q PCA-6, COBYLA 100i", 0.525, 0.097, 0.540, 0.164),
    ("  VQC 10q PCA-10, COBYLA 150i", 0.542, 0.097, 0.518, 0.164),
    ("  VQC 10q top-10, ZZ+RA linear, 300i", 0.793, 0.203, 0.478, 0.285),
    ("  Quantum Kernel SVM (PCA-6)", 0.908, 0.111, 0.009, 0.017),
    ("", None, None, None, None),
    ("QUANTUM — DOMAIN-INFORMED (this notebook)", None, None, None, None),
    ("  A: WildfireQCircuit", results_A['accuracy'], results_A['precision'], results_A['recall'], results_A['f1']),
    ("  B: ZZ+RA linear (reordered features)", results_B['accuracy'], results_B['precision'], results_B['recall'], results_B['f1']),
    ("  C: Random entanglement (ablation)", results_C['accuracy'], results_C['precision'], results_C['recall'], results_C['f1']),
]

for row in all_results:
    name = row[0]
    if row[1] is None:
        print(f"\n{name}")
    else:
        print(f"{name:<50s} {row[1]:>6.3f} {row[2]:>6.3f} {row[3]:>6.3f} {row[4]:>6.3f}")

In [ ]:
# === Cell 12: Resource Requirements Summary (for PDF submission) ===

print("="*65)
print("  QUANTUM RESOURCE REQUIREMENTS — WildfireQCircuit")
print("="*65)
print(f"""\n
Platform:          IBM Qiskit {__import__('qiskit').__version__}
Simulator:         Qiskit Aer (statevector, exact simulation)
Qubits:            {N_QUBITS}

FEATURE MAP (custom WildfireQCircuit):
  Encoding:        Hadamard + RY(x) per qubit
  Entanglement:    Domain-informed CZ topology
  Layers:          2 (data re-uploading)
  CZ gates/layer:  14 (vs 9 for ZZFeatureMap linear)
  Total gates:     {dict(fm_A.count_ops())}
  Depth:           {fm_A.depth()}

ANSATZ (custom, same topology):
  Rotation gates:  RY (trainable)
  Entanglement:    Same domain-informed CZ topology
  Layers:          3 + final rotation
  Trainable params:{an_A.num_parameters}
  Total gates:     {dict(an_A.count_ops())}
  Depth:           {an_A.depth()}

TRAINING:
  Samples:         {X_bal.shape[0]} (balanced 50/50)
  Optimizer:       COBYLA (maxiter=300)
  Training time:   {time_A/60:.1f} minutes

QUBIT-FEATURE MAPPING:
  q0: prior_total_acres           [Fire History]
  q1: prior_max_acres             [Fire History]
  q2: Num High Fire Risk Exposure  [Fire History — HUB NODE]
  q3: min_precip                  [Weather/Moisture]
  q4: Earned Premium              [Insurance]
  q5: CAT CovA Fire Claims        [Insurance]
  q6: Avg PPC                     [Insurance]
  q7: renter_occupied_units       [Socioeconomic]
  q8: education_bachelor          [Socioeconomic]
  q9: housing_vacancy             [Socioeconomic]
""")